<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# 💻 PID for Lateral Control 

In the previous activities we have built up the tools needed to deploy a PID controller. We have used as control signal the angular speed ($\omega$) and as controlled variable the heading of the Duckiebot ($\theta$).

In this exercise you will use what you have learned so far to write a PID controller to command the angular speed of your Duckiebot, to compensate an initial lateral position offset of $0.2 \text{m}$. 

Your robot will have to rely on its own odometry to estimate its current position.

**Note**: the function(s) used to compute the odometry are provided for you here.

May the duck be with you!

## Implement PID controller for lateral control

Implement the function `OffsetControl` inside the file [pid_controller.py](../packages/solution/pid_controller.py).

The controller you are asked to write performs PID control on the $y$ coordinate (in the world frame) of the Duckiebot. It receives the following inputs:

    v_ref:      linear Duckiebot speed.
    y_ref:      reference heading pose.
    theta_curr: the current estimated heading.
    y_curr:     the current estimated "y" coordinate (offset)
    delta_t:    time interval since last call.

and it will produce the following outputs:

    v:       linear velocity of the Duckiebot
    omega:   angular velocity of the Duckiebot


### Sanity Check

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../packages/tests/')

from unit_test import UnitTestPositionPID

from solution.pid_controller import PIDController

# Write here the values of the kinematics parameters you calculated during the activities.
# This test will give you an idea of the behavior of the controller you wrote above.
# Try to play with the parameters, change R, the baseline all the PID variables,
# What change in the resulting graphs? Why?

# Note: this sanity check is neither a prerequisite for completing the exercise nor an instrument for tuning your controller. 

R = 0.0318
baseline = 0.1
gain = 0.6
trim = 0.0
v_0 = 0.2
y_ref = 0.2

# unit test input R, baseline, v_0, gain, trim, PIDController
unit_test = UnitTestPositionPID(R, baseline, v_0, y_ref, gain, trim, PIDController) 
unit_test.test()


# PID tuning

In order to tune the PID two approaches of increasing fidelity are possible:
1. Kinematic model $\dot{q}=Aq$: only simulating the kinematics of the Duckiebot
2. Dynamics model $F=ma$: simulate the dynamics of the Duckiebot, with forces and inertia.

### Tuning scenario
You will tune the PID in a scenario similar to the one in the first AIDO challenge of this learning experience:

- **Starting pose:** $x_0,y_0,\theta_0=[0 m, 0.2 m, 0°]$
- **Commanded velocity:** $v_0=0.22m/s$
- **Goal**: getting to $y_{ref}=0.0m$

This is equivalent to starting in the wrong lane and try following the centerline of the correct lane.

In [ ]:
# Initial conditions of the duckiebot
x0 = 0.0
y0 = 0.2
theta_0 = 0.0 # degrees

initial_pose =  [x0,y0,theta_0]
initial_vel =   [0.0, 0]

# Goal:
y_ref = 0

# Kinematic model
In this part of the notebook you will tune the PID on a kinematic model of the Duckiebot. This is the simplest model of the robot and it will be your first step to get an idea of the values.

In [ ]:
import matplotlib.pyplot as plt

from utils.writer import load_gains, update_gains

def plot_xy_from_lists(xs,ys,y_ref):
    # Plot (x,y) position
    ax = plt.subplot(313)
    plt.grid('on')
    plt.plot(xs, ys)

    plt.axhline(y = y_ref, color = 'r', linestyle = '--')

    plt.xlabel('x', fontsize=12)
    plt.ylabel('y', fontsize=12)

    # Set grass background
    ax.set_facecolor('g')

    top_line=0.2
    bottom_line=-0.2

    # Draw road
    ax.axhspan(ymin=bottom_line,ymax=top_line,facecolor='k')
    plt.axhline(y=top_line,linestyle='-',color='w') # white
    plt.axhline(y=bottom_line,linestyle='-',color='w')
    plt.axhline(y=0.0,linestyle='--',color='#fefe03')

    ax.set_ylim(bottom=-0.3,top=0.3)

    plt.show()


In [ ]:
from simulators import integrate_kinematics
from solution.pid_controller import PIDController

def compute_response(initial_pose, initial_vel = [0,0], y_ref=0,dynamics=integrate_kinematics):
    
    xs, ys, omegas, e_list, angles = dynamics(
        initial_pose,initial_vel,y_ref,controller=PIDController,
        )
        
    plot_xy_from_lists(xs,ys,y_ref)

def tune_gains(kp,kd,ki):
        # write values from sliders to file GAINS.yaml
        update_gains(kp,kd,ki)

        # simulate the controller
        compute_response(initial_pose,y_ref=y_ref,dynamics=integrate_kinematics)

from ipywidgets import interact

interact(tune_gains, kp=5.0,kd=5.0,ki=1.0)

# PID Tuning - Dynamic model
For convenience you can simulate here the dynamic behavior of the robot on the x-y plane without having to run the full simulator. This allows you to tweak the gains of the PID and quickly see the effect on the response of the system.

Let's plot the position of the duckiebot on the $xy$ plane:

In [ ]:
from simulators import integrate_dynamics
from solution.pid_controller import PIDController

def tune_gains(kp,kd,ki):
        # write values from sliders to file OFFSET_GAINS.yaml
        update_gains(kp,kd,ki)

        # simulate the controller
        compute_response(initial_pose,y_ref=y_ref,dynamics=integrate_dynamics)

interact(tune_gains, kp=5.0,kd=5.0,ki=1.0)

# PID Tuning: dynamic model with odometry

Now to get the current pose we use the odometry function you developed. In particular $\hat{y}$ (the estimate of the current $y$ position) is fed to the PID controller and the `dynamics` function now also gets as argument the two functions you developed in the odometry activity (`delta_phi` and `pose_estimation`).

In [ ]:
from odometry import delta_phi, estimate_pose

# We redefine `compute_response` to use the odometry functions as well

def compute_response(initial_pose, initial_vel = [0,0], y_ref=0,dynamics=integrate_kinematics):
    
    xs, ys, omegas, e_list, angles = dynamics(
        initial_pose,initial_vel,y_ref,controller=PIDController,odometry_function=estimate_pose,delta_phi=delta_phi
        )
        
    plot_xy_from_lists(xs,ys,y_ref)

def tune_gains(kp,kd,ki):
        # write values from sliders to file GAINS.yaml
        update_gains(kp,kd,ki)

        # simulate the controller
        compute_response(initial_pose,y_ref=y_ref,dynamics=integrate_dynamics)

interact(tune_gains, kp=5.0,kd=5.0,ki=1.0)

You see that there's a difference in the behavior of the robot compared to the dynamics simulator _without odometry_.

*   Can you explain what is it?
*   What do you think might be the cause of this difference?
*   How could you check if your hypothesis is correct?

### 💻 Test the controller in simulation

Follow [the instructions in the README](../README.md) to create a virtual robot, start the Duckiematrix, bring up the ROS 2 stack on the robot, then build and run your code:

    dts code build -R ROBOT_NAME
    dts code workbench -m -R ROBOT_NAME

The workbench starts two ROS 2 nodes in the robot's namespace: `encoder_pose_node` (provided by the recipe, it estimates the pose from the wheel encoders) and your `pid_controller_node`. You will see their logs in the terminal where you launched the workbench. Nothing should be moving yet.

Instead of a desktop with windows, you drive the nodes from this notebook. The cells below:

- connect this notebook to the robot (`RosSession`) - fill in `ROBOT` first;
- show a panel with the parameters of `pid_controller_node` (press `Send` to apply them, `Read from robot` to see the current values);
- give you `start` / `stop` buttons (`stop` is your "emergency brake": it sends a zero command and resets the odometry);
- plot the trajectory estimated by the odometry, which is what your controller sees.


In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ParamPanel, ActionButtons, TrajectoryPlot

ROBOT = "ROBOT_NAME"    # <-- the name of your (virtual or real) Duckiebot
ros = RosSession(ROBOT)  # connects to the robot's zenoh router and spins in the background


In [ ]:
# Parameters of the controller: set `activity` to pid_offset, move the sliders, then press `Send`.
# The node applies them immediately, also while the robot is driving.
panel = ParamPanel(
    ros, "pid_controller_node",
    {
        "activity": None,                       # text field: pid_offset for this notebook
        "kp": (0.0, 20.0, 0.1),
        "ki": (0.0, 5.0, 0.05),
        "kd": (0.0, 5.0, 0.05),
        "v_ref": (0.0, 0.5, 0.01),              # fixed linear speed [m/s]
        "offset_ref": (-0.3, 0.3, 0.01),        # reference lateral position y [m]
    },
)
# `start` runs your controller, `stop` is the emergency brake (zero command + odometry reset)
buttons = ActionButtons(ros, "pid_controller_node", ["start", "stop"])


In [ ]:
# The robot's belief of its own pose (dead reckoning from the wheel encoders):
# the x/y path and an arrow for the heading. Press `stop` (or `reset` below) to start a new path from (0, 0).
trajectory = TrajectoryPlot(ros, "encoder_pose_node/pose")
reset = ActionButtons(ros, "encoder_pose_node", ["reset"])


To initialize the testing of your controller, set `activity` to `pid_offset`, insert the gains, `offset_ref` (in **meters**) and `v_ref` (between 0 and 0.5 m/s), press `Send` and then `start`. The controller you designed above will then start computing the $\omega$ and the robot will start driving. 

Good example values for this activity are `offset_ref = 0.2` and `v_ref = 0.2`, but you can play with these parameters at runtime (change them and press `Send` again) and see how your PID controller performs. If you run too far off the road, press `stop` and use the joystick to manually drive your robot back to the road. 

At any point you can press the `stop` button to "emergency brake" your Duckiebot. 

<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid-lateral-sim-good.png" alt="pid-lateral-sim-1" style="width: 300px;"/>
  <p>Simulation, lateral PID control. 

$$v_{0} = 0.2, y_{ref} = [0.2, -0.1]$$
</p>
  </div>
</figure>

To test different solutions, change the `OffsetControl` function you wrote, save this file (`Ctrl-S`) and re-build and re-run (`dts code build` and `dts code workbench`). The panels above keep working across a restart of the workbench; re-run their cells if you restarted the notebook kernel.

**Debugging**: a terminal inside the code editor talks to the same robot (start it with `VEHICLE_NAME=ROBOT_NAME bash` so the `ros2` tools connect to the robot's router; lists can take a few seconds to fill: `ros2 topic list --no-daemon --spin-time 5`): `ros2 node list`, `ros2 topic echo /ROBOT_NAME/encoder_pose_node/pose`, `ros2 param list /ROBOT_NAME/pid_controller_node`, `ros2 param set /ROBOT_NAME/pid_controller_node kp 2.0`, `ros2 service call /ROBOT_NAME/pid_controller_node/stop std_srvs/srv/Trigger`.

### 🚙 Test the controller on your Duckiebot

You can follow the same procedure as above to test your code on your real Duckiebot: bring up the ROS 2 stack on it (see the [README](../README.md)), run `dts code workbench -R ROBOT_NAME` (without `-m`) and use the same panels.

**Note**: we suggest to start at very slow speeds with the physical Duckiebot, and get a hang of the interface first. Going wasted in simulation is just a matter of re-setting the instance. Having the Duckiebot go wasted in the physical world might be significantly more time consuming!


<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid-lateral-real-good.png" alt="pid-lateral-real-1" style="width: 300px;"/>
  <p>Duckiebot, lateral PID control. 
  
  $$v_{0} = 0.2, y_{ref} = [0.2, -0.2, 0.2]$$
  </p>
  </div>
</figure>

As before, change the `OffsetControl` function you wrote, save this file (`Ctrl-S`) and re-build and re-run. 


Congratulations, you have completed the control exercise! 